In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

import pyfredapi as pf
import pandas as pd
from fred_api_key import FRED_API_KEY
from time import sleep

API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [ ]:
to_extract_fred = [
    # Moody's Seasoned Aaa Corporate Bond Yield
    # Percent, Daily, Not Seasonally Adjusted
    "DAAA",
    # Federal Funds Effective Rate
    # Percent, Daily, Not Seasonally Adjusted
    "DFF",

    # Crude Oil Prices: West Texas Intermediate (WTI) - Cushing, Oklahoma
    # Dollars per Barrel, Daily, Not Seasonally Adjusted
    "DCOILWTICO",

    # Sticky Price Consumer Price Index less Food and Energy
    # Percent Change from Year Ago, Monthly, Seasonally Adjusted
    "CORESTICKM159SFRBATL",

    # Treasury Yield (2Y)
    # Percent, Daily, Not Seasonally Adjusted
    "DGS2",              # Subtract dgs10 from dgs2 to obtain spread
    # Treasury Yield (10Y)
    # Percent, Daily, Not Seasonally Adjusted
    "DGS10",

    # GDP/GNP; Billions of Dollars, Quarterly, Seasonally Adjusted Annual Rate
    "GDP",

    # Unemployment Rate; Percent, Monthly, Seasonally Adjusted
    "UNRATE",

    # Industrial Production: Total Index
    # Index 2017=100, Monthly, Seasonally Adjusted
    "INDPRO",

    # Consumer Sentiment
    # Index 1966:Q1=100, Monthly, Not Seasonally Adjusted
    "UMCSENT",

    # Bank Credit, All Commercial Banks
    # Billions of U.S. Dollars, Weekly, Seasonally Adjusted
    "TOTBKCR",

    # Import Price Index (End Use): Nonmonetary Gold
    # Index Dec 2024=100, Monthly, Not Seasonally Adjusted
    "IR14270",
]

In [ ]:
from functools import reduce
# from time import sleep

START_DATE = "1999-12-25"
END_DATE = "2026-06-30"

def load_fred_series(series_id: str, api_key: str) -> pd.DataFrame:
    df = pf.get_series(
        series_id=series_id,
        observation_start=START_DATE,
        observation_end=END_DATE,
        api_key=api_key,
    )

    df = (
        df[["date", "value"]]
        .rename(columns={"value": series_id})
        .assign(date=lambda x: pd.to_datetime(x["date"]))
        .set_index("date")
        .sort_index()
    )

    # Add rate of change
    df[f"{series_id}_pct_change"] = df[series_id].pct_change(fill_method=None)

    return df


macro_frames = []

for ticker in to_extract_fred:
    print(f"Downloading {ticker}...")
    macro_frames.append(load_fred_series(ticker, api_key = API_KEY))
    sleep(0.5)

macro_data = reduce(
    lambda left, right: left.join(right, how="outer"),
    macro_frames,
)

daily_index = pd.date_range(START_DATE, END_DATE, freq="D")

macro_data = (
    macro_data
    .reindex(daily_index)
    .ffill()
)

macro_data.index.name = "date"

macro_data.to_csv(CACHE_PATH / "macro_data.csv")

print(macro_data.info())
print(macro_data.head())

: 